# Decoding Customer Value: A SQL-Driven Retention Strategy
## Analytical Notebook — Full Walkthrough

**Summer Projects '26 | Consulting & Analytics Club, IIT Guwahati**

This notebook documents the complete analytical workflow for this project.
Every step is explained with business context before the code.

---

**Dataset:** 3,900 customer records from a D2C fashion brand  
**Problem:** Is the brand building genuine loyalty or discount-dependent buying?  
**Constraint:** No loyalty score, no churn label, no timestamps — everything must be constructed.


## 1. Problem Statement

A D2C fashion brand sells Clothing, Accessories, Footwear, and Outerwear across the US.
It runs a promotional program but has never analyzed whether those discounts are creating loyal customers
or training buyers to only purchase when deals are available.

**Five key questions the brand needs to answer:**
1. Who are genuinely loyal customers vs. those who only buy with discounts?
2. What behavioral patterns today predict high customer value over time?
3. Which geographies and demographics are commercially underlevered?
4. How should the brand restructure its promotional strategy to protect margins?
5. What does the ideal customer profile look like?

**The core analytical constraint:**
The dataset has no loyalty score, no churn label, and no timestamps.
Every concept — loyalty, retention, customer value — must be constructed from available behavioral signals.


## 2. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Load raw data
df = pd.read_csv('data/raw/shopping_trends.csv')
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")


## 3. Data Cleaning

The dataset is remarkably clean — no missing values, no duplicates.
We standardize categorical fields, create binary flags, and map frequency strings to numeric annual values.

**Key decisions:**
- `discount_applied` and `promo_code_used` (Yes/No) → binary flags for arithmetic
- `frequency_of_purchases` (Weekly/Monthly/etc.) → annual numeric for frequency calculations


In [ ]:
# Standardize columns
import re
def to_snake(name):
    name = re.sub(r'\(.*?\)', '', name)
    name = name.strip().lower()
    name = re.sub(r'[\s\-]+', '_', name)
    return re.sub(r'[^a-z0-9_]', '', name).strip('_')

df.columns = [to_snake(c) for c in df.columns]

# Standardize categoricals
for c in ['gender','category','location','season','subscription_status',
          'discount_applied','promo_code_used','frequency_of_purchases']:
    df[c] = df[c].str.strip().str.title()

# Binary flags
df['discount_applied_flag'] = (df['discount_applied'] == 'Yes').astype(int)
df['promo_code_used_flag']  = (df['promo_code_used']  == 'Yes').astype(int)
df['is_subscribed']         = (df['subscription_status'] == 'Yes').astype(int)

# Frequency to annual
FREQ_MAP = {'Weekly':52,'Bi-Weekly':26,'Fortnightly':26,'Monthly':12,
            'Quarterly':4,'Every 3 Months':4,'Annually':1,'Bi-Monthly':6}
df['purchase_freq_annual'] = df['frequency_of_purchases'].map(FREQ_MAP).fillna(12)

print("Cleaned dataset:")
print(f"  Rows: {len(df)}, Columns: {len(df.columns)}")
print(f"  Discount applied rate: {df['discount_applied_flag'].mean():.1%}")
print(f"  Promo code used rate:  {df['promo_code_used_flag'].mean():.1%}")
print(f"  Subscription rate:     {df['is_subscribed'].mean():.1%}")
df.head(3)


## 4. Feature Engineering

We build customer-level metrics from the transactional data.
Every feature must answer a question the brand actually cares about.

### Customer Value Metrics
- `total_spend_est`: estimated lifetime spend = purchase_amount × (previous_purchases + 1)
- `value_tier`: High/Medium/Low based on 33rd and 66th percentiles of total_spend_est

### Promo Dependency Score
A composite 0–100 score measuring how much a customer's purchasing depends on promotions.

### Retention Proxy Score
⚠️ **Important limitation:** No timestamps means we cannot calculate true retention.
We construct a defensible proxy using purchase history, frequency, satisfaction, subscription status, and low promo dependency.


In [ ]:
fe = df.copy()

# Value metrics
fe['total_spend_est'] = fe['purchase_amount'] * (fe['previous_purchases'] + 1)
p33 = fe['total_spend_est'].quantile(0.33)
p66 = fe['total_spend_est'].quantile(0.66)
fe['value_tier'] = pd.cut(fe['total_spend_est'], bins=[-np.inf, p33, p66, np.inf],
                           labels=['Low','Medium','High'])

# Promo dependency score
purchase_norm = fe['previous_purchases'] / fe['previous_purchases'].max()
fe['promo_dependency_score'] = (
    0.40 * fe['discount_applied_flag'] +
    0.40 * fe['promo_code_used_flag'] +
    0.20 * (1 - purchase_norm)
) * 100

# Retention proxy score (NOT true retention)
scaler = MinMaxScaler()
prev_n  = scaler.fit_transform(fe[['previous_purchases']])[:,0]
freq_n  = scaler.fit_transform(fe[['purchase_freq_annual']])[:,0]
rate_n  = scaler.fit_transform(fe[['review_rating']])[:,0]
fe['retention_proxy_score'] = (
    0.35 * prev_n + 0.25 * freq_n + 0.20 * rate_n +
    0.10 * fe['is_subscribed'] + 0.10 * (1 - fe['promo_dependency_score']/100)
) * 100

print(f"Value tier distribution:")
print(fe['value_tier'].value_counts().to_string())
print(f"\nRetention proxy: {fe['retention_proxy_score'].min():.1f} – {fe['retention_proxy_score'].max():.1f}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fe['value_tier'].value_counts().reindex(['High','Medium','Low']).plot(kind='bar', ax=ax1, color=['#1a2e4a','#4472C4','#D9D9D9'])
ax1.set_title('Customer Value Tier Distribution')
ax1.set_xlabel('Value Tier')
ax1.set_ylabel('Count')
ax1.tick_params(rotation=0)
fe['retention_proxy_score'].hist(bins=30, ax=ax2, color='#4472C4', edgecolor='white')
ax2.set_title('Retention Proxy Score Distribution')
ax2.set_xlabel('Score (0–100)')
ax2.set_ylabel('Count')
plt.tight_layout()
plt.show()


## 5. Loyalty Definition 1: Behavioral Loyalty

**Hypothesis:** A loyal customer buys repeatedly, spends well, is satisfied, and doesn't rely on discounts.

**Variables:** purchase_count (proxy for frequency), total_spend_est, review_rating, promo_usage_rate  
**Weights:** 30% purchase frequency + 30% spend + 20% satisfaction + 20% low promo usage


In [ ]:
fe['purchase_count'] = fe['previous_purchases'] + 1
fe['promo_usage_rate'] = (fe['discount_applied_flag'] + fe['promo_code_used_flag']) / 2

s1 = pd.DataFrame(
    scaler.fit_transform(fe[['purchase_count','total_spend_est','review_rating']]),
    columns=['pc_n','spend_n','rating_n']
)
fe['loyalty_score_1'] = (
    0.30 * s1['pc_n'] +
    0.30 * s1['spend_n'] +
    0.20 * s1['rating_n'] +
    0.20 * (1 - fe['promo_usage_rate'])
)
ls1_q33 = fe['loyalty_score_1'].quantile(0.33)
ls1_q66 = fe['loyalty_score_1'].quantile(0.66)
fe['loyalty_def1'] = pd.cut(fe['loyalty_score_1'], bins=[-np.inf,ls1_q33,ls1_q66,np.inf],
                             labels=['Low Loyalty','Medium Loyalty','High Loyalty'])

grp1 = fe.groupby('loyalty_def1', observed=True).agg(
    n=('purchase_count','count'),
    avg_ltv=('total_spend_est','mean'),
    avg_promo=('promo_dependency_score','mean'),
    avg_rating=('review_rating','mean'),
    avg_retention=('retention_proxy_score','mean'),
).round(2)
print("Loyalty Definition 1 (Behavioral) — segment profiles:")
print(grp1)


## 6. Loyalty Definition 2: Commercial Loyalty

**Hypothesis:** A commercially loyal customer makes an economic commitment — high spend, high AOV, frequent purchasing, subscription, and doesn't rely on discounts to transact.

**Variables:** total_spend_est, avg_order_value, purchase_freq_annual, is_subscribed, discount_applied_flag (inverted)  
**Weights:** 30% spend + 25% AOV + 20% frequency + 15% subscription + 10% no discount


In [ ]:
s2 = pd.DataFrame(
    scaler.fit_transform(fe[['total_spend_est','purchase_amount','purchase_freq_annual']]),
    columns=['spend_n','aov_n','freq_n']
)
fe['loyalty_score_2'] = (
    0.30 * s2['spend_n'] + 0.25 * s2['aov_n'] + 0.20 * s2['freq_n'] +
    0.15 * fe['is_subscribed'] + 0.10 * (1 - fe['discount_applied_flag'])
)
ls2_q33 = fe['loyalty_score_2'].quantile(0.33)
ls2_q66 = fe['loyalty_score_2'].quantile(0.66)
fe['loyalty_def2'] = pd.cut(fe['loyalty_score_2'], bins=[-np.inf,ls2_q33,ls2_q66,np.inf],
                             labels=['Low Loyalty','Medium Loyalty','High Loyalty'])

grp2 = fe.groupby('loyalty_def2', observed=True).agg(
    n=('purchase_count','count'),
    avg_ltv=('total_spend_est','mean'),
    avg_promo=('promo_dependency_score','mean'),
    avg_rating=('review_rating','mean'),
    avg_retention=('retention_proxy_score','mean'),
).round(2)
print("Loyalty Definition 2 (Commercial) — segment profiles:")
print(grp2)


## 7. Loyalty Definition Comparison

We evaluate both definitions using four monotonicity tests:
A well-constructed loyalty score should produce High Loyalty > Medium Loyalty > Low Loyalty
on all four key metrics: revenue, promo dependency (inverse), satisfaction, and retention signal.

**Why Definition 1 is selected:**
- Both score 4/4 on monotonicity, but Definition 1 shows much wider separation between tiers on promo dependency (55 point gap vs. 5 point gap for Definition 2)
- Definition 1 directly incorporates satisfaction (review_rating), which Definition 2 ignores
- Definition 1 captures behavioral attachment; Definition 2 can confuse high-frequency bargain hunters with loyal customers


In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Revenue (High > Low)', 'Promo Dep (High < Low)', 'Satisfaction (High > Low)', 'Retention (High > Low)'],
    'Def 1 (Behavioral)': [
        f"${grp1.loc['High Loyalty','avg_ltv']:.0f} vs ${grp1.loc['Low Loyalty','avg_ltv']:.0f}",
        f"{grp1.loc['High Loyalty','avg_promo']:.1f} vs {grp1.loc['Low Loyalty','avg_promo']:.1f}",
        f"{grp1.loc['High Loyalty','avg_rating']:.2f} vs {grp1.loc['Low Loyalty','avg_rating']:.2f}",
        f"{grp1.loc['High Loyalty','avg_retention']:.1f} vs {grp1.loc['Low Loyalty','avg_retention']:.1f}",
    ],
    'Def 2 (Commercial)': [
        f"${grp2.loc['High Loyalty','avg_ltv']:.0f} vs ${grp2.loc['Low Loyalty','avg_ltv']:.0f}",
        f"{grp2.loc['High Loyalty','avg_promo']:.1f} vs {grp2.loc['Low Loyalty','avg_promo']:.1f}",
        f"{grp2.loc['High Loyalty','avg_rating']:.2f} vs {grp2.loc['Low Loyalty','avg_rating']:.2f}",
        f"{grp2.loc['High Loyalty','avg_retention']:.1f} vs {grp2.loc['Low Loyalty','avg_retention']:.1f}",
    ],
    'Pass/Fail Def 1': ['✓','✓','✓','✓'],
    'Pass/Fail Def 2': ['✓','✓','✓','✓'],
})
print("Loyalty Definition Comparison:")
print(comparison.to_string(index=False))
print("\n→ Both pass 4/4 but Def 1 shows far wider promo dependency separation.")
print("→ SELECTED: Definition 1 (Behavioral Loyalty)")


## 8. Final Customer Segmentation

We combine value tier, loyalty segment, promo segment, and satisfaction to create six actionable segments.
Every segment is defined by explicit variable conditions — no vague labels.


In [ ]:
# Load full featured data from pipeline output
fe_full = pd.read_csv('data/features/customer_features.csv')
seg_counts = fe_full['final_segment'].value_counts()
print("Final Segment Distribution:")
for seg, cnt in seg_counts.items():
    print(f"  {seg:<35} {cnt:>5} ({cnt/len(fe_full)*100:.1f}%)")

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#1a2e4a','#E67E22','#E74C3C','#27AE60','#7F8C8D','#BDC3C7']
seg_counts.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Number of Customers')
ax.set_title('Customer Segments by Size')
plt.tight_layout()
plt.show()

# Segment definitions
definitions = {
    'Loyal High-Value': 'value_tier=High + High Loyalty + promo_segment ∈ {Organic, Selective}',
    'High-Value Promo-Dependent': 'value_tier=High + promo_segment ∈ {Promo Dependent, Bargain Hunter}',
    'At-Risk Dissatisfied': 'satisfaction_tier=Dissatisfied + value_tier ∈ {High, Medium}',
    'Organic Mid-Value': 'value_tier=Medium + promo_segment ∈ {Organic, Selective}',
    'Low-Repeat Bargain Hunter': 'value_tier=Low + promo_segment ∈ {Promo Dep, Bargain Hunter} + prev_purchases ≤ 5',
    'Low-Value Low-Engagement': 'All remaining customers',
}
print("\nSegment Definitions:")
for seg, defn in definitions.items():
    print(f"  {seg}: {defn}")


## 9. SQL Analysis Overview

The full SQL analysis lives in `sql/analysis_queries.sql`.
Here we execute the key queries against the SQLite database to show the outputs.


In [ ]:
conn = sqlite3.connect('sql/customer_intelligence.db')

print("=== Q1: Loyal vs. Discount-Driven ===")
q1 = pd.read_sql('''
SELECT final_segment, COUNT(*) AS n,
       ROUND(AVG(purchase_amount),2) AS avg_order_val,
       ROUND(AVG(total_spend_est),0) AS avg_ltv,
       ROUND(AVG(promo_dependency_score),1) AS promo_score,
       ROUND(AVG(retention_proxy_score),1) AS retention_proxy
FROM customers GROUP BY final_segment ORDER BY avg_ltv DESC
''', conn)
print(q1.to_string(index=False))

print("\n=== Revenue: Organic vs Promo-Using ===")
q_rev = pd.read_sql('''
SELECT CASE WHEN promo_usage_rate>0 THEN "Promo-Using" ELSE "Organic" END AS type,
       COUNT(*) AS n, ROUND(SUM(total_spend_est),0) AS revenue_est,
       ROUND(SUM(total_spend_est)*100.0/(SELECT SUM(total_spend_est) FROM customers),1) AS pct
FROM customers GROUP BY type
''', conn)
print(q_rev.to_string(index=False))
conn.close()


## 10. Dashboard Tables

Four Power BI-ready tables produced by the pipeline.


In [ ]:
pyramid = pd.read_csv('data/dashboard/customer_pyramid.csv')
print("Customer Pyramid:")
print(pyramid[['value_tier','customer_count','pct_of_customers','pct_of_revenue','avg_spend','avg_retention']].to_string(index=False))

print("\nPromo Dependency vs Retention:")
pr = pd.read_csv('data/dashboard/promo_dependency_retention.csv')
print(pr[['final_segment','avg_promo_dependency','avg_retention_proxy','customer_count']].to_string(index=False))

print("\nCategory Summary:")
cs = pd.read_csv('data/dashboard/category_summary.csv')
print(cs[['category','avg_prev_purchases','avg_promo_dependency','avg_rating','category_role']].to_string(index=False))


## 11. Business Recommendations

### Promotional Sunset Plan (4 phases)

| Phase | Segment | Action | Timeline | Risk |
|-------|---------|--------|----------|------|
| 1 | Loyal High-Value (599) | Remove all blanket promos immediately | Weeks 1–4 | Near-zero |
| 2 | HV Promo-Dependent (450) | Stage down discount depth → value-adds → remove | Weeks 5–16 | Medium |
| 3 | Low-Repeat Bargain Hunter (424) | Stop promo targeting, redirect budget | Weeks 8–20 | Low |
| 4 | Organic Mid-Value (557) | Nurture upward: cross-sell, subscription offers | Weeks 4–24 | Low |

### Ideal Customer Profile
Age 30–57, slightly female, Clothing + Accessories, $76/order, 38 prior purchases, 4.07/5 rating, near-zero promo dependency, $2,937 avg LTV.

### Acquisition Strategy
Create lookalike audience from the 599 Loyal High-Value customer IDs. Target High Opportunity states (Montana, Alaska, Alabama, Idaho, Michigan). Creative message: quality and style, NOT discounts.


## 12. Conclusion

**Is the brand building loyalty or buying transactions?**

**Both — but the healthy half is larger.**

56.8% of estimated revenue comes from organic buyers who never need promotions.
The 599 Loyal High-Value customers alone drive 28.6% of revenue from 15.4% of the base.

The promotional program is mostly funding two groups:
1. High-Value Promo-Dependent customers (21.2% of revenue) who spend well but may only buy with discounts
2. Low-Repeat Bargain Hunters (1.7% of revenue) who represent wasted promotional spend

**The path forward:**
1. Protect the organic loyal core with recognition, not discounts
2. Run a controlled 12-week test to see if the high-value promo-dependent segment can be transitioned
3. Redirect bargain-hunter promotional budget to nurturing the Organic Mid-Value pipeline
4. Build a VIP/subscription offer that converts the 0%-subscribed Loyal High-Value segment

The brand is not in a crisis. It is at an inflection point where deliberate action can shift the customer mix toward organic loyalty — or continued inaction can cement a discount-dependent cycle.
